In [1]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
import time
import re

pd.set_option('display.max_columns', 200) 

In [ ]:


def scrape_euroleague_json(year):
    url = f'https://www.eurobasket.com/Euroleague/basketball-Players.aspx?Year={year}'
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    response = requests.get(url, headers=headers)

    # Check if request was successful
    if response.status_code != 200:
        print(f"Failed to retrieve data for {year}. HTTP Status Code: {response.status_code}")
        return []

    # Parse HTML content
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find script tag containing JSON data
    script_tag = soup.find('script', text=lambda t: t and 'strData' in t)
    
    if script_tag is None:
        print(f"No script tag found for year {year}")
        return []

    # Extract JSON string using regex (more stable)
    match = re.search(r"strData='(.*?)';", script_tag.string)
    
    if not match:
        print(f"Failed to extract JSON data for year {year}")
        print("Page source:", script_tag.string[:300])  # Log a snippet of the page source for inspection
        return []

    json_str = match.group(1)  # Extract matched JSON string

    try:
        player_data = json.loads(json_str)
    except json.JSONDecodeError:
        print(f"Error decoding JSON for year {year}.")
        return []

    return player_data

# Start timer
start_time = time.time()

# Define seasons (exclude 2020 covid season)
years = [ 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021]

all_players = []

for year in years:
    print(f"Scraping data for {year}...")
    players = scrape_euroleague_json(year)
    
    if players:
        all_players.extend(players)
    
    time.sleep(5)  # delay between requests

# Convert to DataFrame
df = pd.DataFrame(all_players)

# End timer
end_time = time.time()
total_time = end_time - start_time

print(f"\n Scraped data for {len(years)} seasons, {len(df)} players.")
print(f"⏱️ Total execution time: {total_time:.2f} seconds.")

df.to_csv("euroleague_players_2008_2021.csv", index=False)

df.head().T



In [2]:
df = pd.read_csv('euroleague_players_2008_2021.csv').copy()
df.head()

,PLAYERID,PLAYERNAME,TEAMNAME,TEAMCOUNTRY,TEAMNO,COLLEGE,COLLEGESTART,HEIGHT,HEIGHTIN,SKILLS,AGE,NAT,POSITION,NAT1,LEAGUE,LEAGUENAME,FROM,NBADraft,NCAA,Season,Games,MIN,PTS,FGPM2,FGPA2,FGPM3,FGPA3,FTM,FTA,REBO,REBD,REBT,AS,PF,PFRV,BS,BSAG,ST,TO,MVP
0,94777,Aboubakar Zaki Amadou,Nancy,NaN,421,1,0,214.0,7.0,2.5,31.0,IMPORT,C,CMR,1,ProB,19,0,0,2008-2009,10,8.4,1.3,0.5,1.1,0.0,0.1,0.3,0.9,0.6,1.4,2.0,0.2,1.3,0.0,0.5,0.0,0.1,0.0,0.0
1,47036,Akingbala Akinlolu,Nancy,NaN,421,228,0,208.0,6.1,2.5,34.0,IMPORT,C,NGR,1,ProB,17,0,0,2008-2009,2,17.5,8.0,3.5,5.5,0.0,0.0,1.0,2.5,1.5,2.5,4.0,0.0,2.0,0.0,1.5,0.0,0.5,0.0,0.0
2,56913,Akyol Cenk,Anadolu Efes,NaN,99,1,0,198.0,6.6,1.5,32.0,IMPORT,G/F,TUR,1,BSL,19,0,0,2008-2009,6,8.2,3.3,0.7,1.0,0.5,1.8,0.5,0.7,0.3,0.2,0.5,0.3,1.0,0.0,0.0,0.0,0.5,0.0,0.0
3,80014,Alana Sheila,Avenida,NaN,7639,1,0,191.0,6.3,3.5,30.0,IMPORT,C,ESP,1,LFB,12,0,0,2008-2009,17,1.1,0.5,0.0,0.1,0.1,0.4,0.1,0.1,0.0,0.0,0.0,0.0,0.1,0.0,0.1,0.0,0.0,0.0,0.0
4,154742,Albicy Tracy,Villeneuve,NaN,7605,1,0,157.0,5.2,1.5,20.0,IMPORT,PG,FRA,1,LFB,12,0,0,2008-2009,8,0.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4371 entries, 0 to 4370
Data columns (total 40 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PLAYERID      4371 non-null   int64  
 1   PLAYERNAME    4371 non-null   object 
 2   TEAMNAME      4371 non-null   object 
 3   TEAMCOUNTRY   0 non-null      float64
 4   TEAMNO        4371 non-null   int64  
 5   COLLEGE       4371 non-null   int64  
 6   COLLEGESTART  4371 non-null   int64  
 7   HEIGHT        4369 non-null   float64
 8   HEIGHTIN      4369 non-null   float64
 9   SKILLS        4371 non-null   float64
 10  AGE           4370 non-null   float64
 11  NAT           4371 non-null   object 
 12  POSITION      4369 non-null   object 
 13  NAT1          4371 non-null   object 
 14  LEAGUE        4371 non-null   int64  
 15  LEAGUENAME    4371 non-null   object 
 16  FROM          4371 non-null   int64  
 17  NBADraft      4371 non-null   int64  
 18  NCAA          4371 non-null 

In [4]:
df.columns

Index(['PLAYERID', 'PLAYERNAME', 'TEAMNAME', 'TEAMCOUNTRY', 'TEAMNO',
       'COLLEGE', 'COLLEGESTART', 'HEIGHT', 'HEIGHTIN', 'SKILLS', 'AGE', 'NAT',
       'POSITION', 'NAT1', 'LEAGUE', 'LEAGUENAME', 'FROM', 'NBADraft', 'NCAA',
       'Season', 'Games', 'MIN', 'PTS', 'FGPM2', 'FGPA2', 'FGPM3', 'FGPA3',
       'FTM', 'FTA', 'REBO', 'REBD', 'REBT', 'AS', 'PF', 'PFRV', 'BS', 'BSAG',
       'ST', 'TO', 'MVP'],
      dtype='object')

In [5]:
# drop unnecessary columns

df = df[[ 'PLAYERNAME', 'TEAMNAME', 'HEIGHT', 'HEIGHTIN', 
        'POSITION', 'NAT1', 'Season', 'Games', 'MIN', 'PTS', 
         'REBT', 'AS', 'BS', 'ST', 'TO']]

In [6]:
df.isna().sum()

PLAYERNAME    0
TEAMNAME      0
HEIGHT        2
HEIGHTIN      2
POSITION      2
NAT1          0
Season        0
Games         0
MIN           0
PTS           0
REBT          0
AS            0
BS            0
ST            0
TO            0
dtype: int64

In [7]:
df_filtered = df[df['HEIGHT'].isna()]
df_filtered

,PLAYERNAME,TEAMNAME,HEIGHT,HEIGHTIN,POSITION,NAT1,Season,Games,MIN,PTS,REBT,AS,BS,ST,TO
226,Lopez Vinuesa,Unicaja,NaN,NaN,NaN,ESP,2008-2009,2,2.0,2.5,0.0,0.5,0.0,0.0,0.0
228,Losco Lisa,Familia Schio,NaN,NaN,NaN,ITA,2008-2009,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0


It seems missing values belong to players who had small chances in the league.

In [8]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PLAYERNAME,4371,1939,Llull Sergio,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TEAMNAME,4371,101,Olympiacos,191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HEIGHT,4369.0,NaN,NaN,NaN,199.333028,9.020713,157.0,193.0,200.0,206.0,230.0
HEIGHTIN,4369.0,NaN,NaN,NaN,6.492948,0.375515,5.1,6.2,6.5,6.8,7.7
POSITION,4369,11,C,762,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NAT1,4371,84,USA,1080,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Season,4371,12,2011-2012,494,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Games,4371.0,NaN,NaN,NaN,14.05262,10.261678,1.0,5.0,12.0,22.0,41.0
MIN,4371.0,NaN,NaN,NaN,16.688698,8.112042,0.0,10.5,17.4,23.0,37.3
PTS,4371.0,NaN,NaN,NaN,6.336948,4.281225,0.0,3.0,6.0,9.2,32.0


There are some players with very small values in Games played. I’ll filter out the players with low Games by removing those below the 25th percentile

In [9]:
df = df[(df['Games'] >= 5)]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3318 entries, 0 to 4370
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PLAYERNAME  3318 non-null   object 
 1   TEAMNAME    3318 non-null   object 
 2   HEIGHT      3318 non-null   float64
 3   HEIGHTIN    3318 non-null   float64
 4   POSITION    3318 non-null   object 
 5   NAT1        3318 non-null   object 
 6   Season      3318 non-null   object 
 7   Games       3318 non-null   int64  
 8   MIN         3318 non-null   float64
 9   PTS         3318 non-null   float64
 10  REBT        3318 non-null   float64
 11  AS          3318 non-null   float64
 12  BS          3318 non-null   float64
 13  ST          3318 non-null   float64
 14  TO          3318 non-null   float64
dtypes: float64(9), int64(1), object(5)
memory usage: 414.8+ KB


In [10]:
df['MIN'].describe()

count    3318.000000
mean       17.810277
std         7.151252
min         0.000000
25%        12.600000
50%        18.400000
75%        23.300000
max        37.300000
Name: MIN, dtype: float64

* there are also 0 values for seasonal average for MIN column

In [11]:
df_filtered = df[df['MIN']== 0]
df_filtered

,PLAYERNAME,TEAMNAME,HEIGHT,HEIGHTIN,POSITION,NAT1,Season,Games,MIN,PTS,REBT,AS,BS,ST,TO
24,Barroilhet Tamzin,Bourges,183.0,6.0,G,FRA,2008-2009,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
38,Besovic Nemanja,Partizan NIS,221.0,7.3,C,SRB,2008-2009,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
99,Erbas Burcu,Fenerbahce,173.0,5.8,G,TUR,2008-2009,9,0.0,0.0,0.0,0.0,0.0,0.0,0.0
113,Firat Duygu,Fenerbahce,189.0,6.3,F,TUR,2008-2009,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
131,Gonen Tugba,Besiktas,170.0,5.7,PG,TUR,2008-2009,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
140,Gulin Nika,Sibenik,173.0,5.8,G,CRO,2008-2009,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
234,Martins Elsa,Bourges,187.0,6.2,F,FRA,2008-2009,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
252,Mirotic Nikola,Real Madrid,207.0,6.1,F/C,MNT,2008-2009,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
365,Stehlikova Jana,USK Praha,192.0,6.4,C,CZE,2008-2009,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
369,Surkusa Liga,TTT Riga,194.0,6.5,C,LAT,2008-2009,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
df = df[df['MIN'] > 0]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3305 entries, 0 to 4370
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PLAYERNAME  3305 non-null   object 
 1   TEAMNAME    3305 non-null   object 
 2   HEIGHT      3305 non-null   float64
 3   HEIGHTIN    3305 non-null   float64
 4   POSITION    3305 non-null   object 
 5   NAT1        3305 non-null   object 
 6   Season      3305 non-null   object 
 7   Games       3305 non-null   int64  
 8   MIN         3305 non-null   float64
 9   PTS         3305 non-null   float64
 10  REBT        3305 non-null   float64
 11  AS          3305 non-null   float64
 12  BS          3305 non-null   float64
 13  ST          3305 non-null   float64
 14  TO          3305 non-null   float64
dtypes: float64(9), int64(1), object(5)
memory usage: 413.1+ KB


In [13]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
HEIGHT,3305.0,199.439032,8.881960,157.0,193.0,200.0,206.0,230.0
HEIGHTIN,3305.0,6.496481,0.373292,5.1,6.2,6.5,6.8,7.7
Games,3305.0,17.896823,8.842615,5.0,10.0,16.0,24.0,41.0
MIN,3305.0,17.880333,7.077332,0.1,12.6,18.4,23.3,37.3
PTS,3305.0,6.886172,3.832321,0.0,3.9,6.6,9.4,23.3
REBT,3305.0,2.743238,1.605766,0.0,1.5,2.5,3.7,10.7
AS,3305.0,1.390590,1.252704,0.0,0.5,1.0,1.9,8.7
BS,3305.0,0.238003,0.314532,0.0,0.0,0.1,0.3,2.6
ST,3305.0,0.596793,0.400773,0.0,0.3,0.5,0.8,2.7
TO,3305.0,0.941331,0.743412,0.0,0.4,0.9,1.4,4.7


After removing outliers for MIN and Games, the dataset dropped by about 24%, from 4371 to 3305 players. This helps focus on players who played a meaningful time in the league.

In [14]:
df.nunique()

PLAYERNAME    1376
TEAMNAME        77
HEIGHT          57
HEIGHTIN        24
POSITION        11
NAT1            76
Season          12
Games           37
MIN            334
PTS            186
REBT            92
AS              75
BS              25
ST              27
TO              41
dtype: int64

In [15]:
df.head()

,PLAYERNAME,TEAMNAME,HEIGHT,HEIGHTIN,POSITION,NAT1,Season,Games,MIN,PTS,REBT,AS,BS,ST,TO
0,Aboubakar Zaki Amadou,Nancy,214.0,7.0,C,CMR,2008-2009,10,8.4,1.3,2.0,0.2,0.5,0.1,0.0
2,Akyol Cenk,Anadolu Efes,198.0,6.6,G/F,TUR,2008-2009,6,8.2,3.3,0.5,0.3,0.0,0.5,0.0
3,Alana Sheila,Avenida,191.0,6.3,C,ESP,2008-2009,17,1.1,0.5,0.0,0.0,0.1,0.0,0.0
4,Albicy Tracy,Villeneuve,157.0,5.2,PG,FRA,2008-2009,8,0.4,0.0,0.0,0.0,0.0,0.0,0.0
8,Alves Audrey,Villeneuve,190.0,6.3,F,FRA,2008-2009,9,0.4,0.0,0.1,0.0,0.0,0.0,0.0


In [16]:
# change the order of first and last names in the PLAYERNAME 
df['PLAYERNAME'] = df['PLAYERNAME'].apply(lambda x: ' '.join(x.split(' ')[::-1]))
df.head()


,PLAYERNAME,TEAMNAME,HEIGHT,HEIGHTIN,POSITION,NAT1,Season,Games,MIN,PTS,REBT,AS,BS,ST,TO
0,Amadou Zaki Aboubakar,Nancy,214.0,7.0,C,CMR,2008-2009,10,8.4,1.3,2.0,0.2,0.5,0.1,0.0
2,Cenk Akyol,Anadolu Efes,198.0,6.6,G/F,TUR,2008-2009,6,8.2,3.3,0.5,0.3,0.0,0.5,0.0
3,Sheila Alana,Avenida,191.0,6.3,C,ESP,2008-2009,17,1.1,0.5,0.0,0.0,0.1,0.0,0.0
4,Tracy Albicy,Villeneuve,157.0,5.2,PG,FRA,2008-2009,8,0.4,0.0,0.0,0.0,0.0,0.0,0.0
8,Audrey Alves,Villeneuve,190.0,6.3,F,FRA,2008-2009,9,0.4,0.0,0.1,0.0,0.0,0.0,0.0


In [17]:
# drop the HEIGHTIN column as it will not be included
df = df.drop(columns=['HEIGHTIN'])

In [18]:
# rename columns 
df = df.rename(columns={
    'PLAYERNAME': 'name',
    'TEAMNAME': 'team',
    'HEIGHT': 'height',
    'POSITION': 'position',
    'NAT1': 'country',
    'MIN': 'minutes',
    'PTS': 'points',
    'REBT': 'rebounds',
    'AS': 'assists',
    'BS': 'blocks',
    'ST': 'steals',
    'TO': 'turnovers'
})

df.tail()

,name,team,height,position,country,Season,Games,minutes,points,rebounds,assists,blocks,steals,turnovers
4366,Denis Zakharov,Zenit,192.0,G,RUS,2020-2021,17,5.0,0.9,0.4,0.9,0.1,0.3,0.6
4367,Paul Zipser,Bayern,203.0,F,GER,2020-2021,39,21.9,9.1,3.0,1.1,0.3,0.7,0.7
4368,Ante Zizic,Maccabi T-A,210.0,C,CRO,2020-2021,34,19.8,9.1,5.4,0.7,0.8,0.4,1.4
4369,Yovel Zoosman,Maccabi T-A,198.0,SF,POL,2020-2021,11,13.5,2.6,1.5,0.6,0.0,0.5,0.5
4370,Andrey Zubkov,Zenit,206.0,PF,RUS,2020-2021,39,16.0,5.1,2.1,1.1,0.0,0.3,0.9


In [19]:
df['position'].unique()

array(['C', 'G/F', 'PG', 'F', 'G', 'C/F', 'F/C', 'F/G', 'SF', 'PF', 'SG'],
      dtype=object)

In [20]:
df['position'].value_counts()

position
C      584
G      533
PG     485
F      441
SF     264
PF     221
SG     195
F/C    192
G/F    175
C/F    123
F/G     92
Name: count, dtype: int64

I will simplify the positioins into three categories: Guard, Forward and Center. I guess this approach better reflects the modern, positionless style of basketball, where players often take on multiple roles

In [21]:
# define mapping for positions
position_mapping = {
    'C': 'Center',
    'G': 'Guard',
    'PG': 'Guard',
    'F': 'Forward',
    'SF': 'Forward',
    'PF': 'Forward',
    'SG': 'Guard',
    'F/C': 'Forward',
    'G/F': 'Guard',
    'C/F': 'Center',
    'F/G': 'Forward'
}

# apply mapping
df['position'] = df['position'].replace(position_mapping)

df.head()

,name,team,height,position,country,Season,Games,minutes,points,rebounds,assists,blocks,steals,turnovers
0,Amadou Zaki Aboubakar,Nancy,214.0,Center,CMR,2008-2009,10,8.4,1.3,2.0,0.2,0.5,0.1,0.0
2,Cenk Akyol,Anadolu Efes,198.0,Guard,TUR,2008-2009,6,8.2,3.3,0.5,0.3,0.0,0.5,0.0
3,Sheila Alana,Avenida,191.0,Center,ESP,2008-2009,17,1.1,0.5,0.0,0.0,0.1,0.0,0.0
4,Tracy Albicy,Villeneuve,157.0,Guard,FRA,2008-2009,8,0.4,0.0,0.0,0.0,0.0,0.0,0.0
8,Audrey Alves,Villeneuve,190.0,Forward,FRA,2008-2009,9,0.4,0.0,0.1,0.0,0.0,0.0,0.0


In [22]:
df['position'].value_counts()

position
Guard      1388
Forward    1210
Center      707
Name: count, dtype: int64

In [23]:
# convert height column to int
df['height'] = df['height'].astype(int)

In [24]:
df.tail()

,name,team,height,position,country,Season,Games,minutes,points,rebounds,assists,blocks,steals,turnovers
4366,Denis Zakharov,Zenit,192,Guard,RUS,2020-2021,17,5.0,0.9,0.4,0.9,0.1,0.3,0.6
4367,Paul Zipser,Bayern,203,Forward,GER,2020-2021,39,21.9,9.1,3.0,1.1,0.3,0.7,0.7
4368,Ante Zizic,Maccabi T-A,210,Center,CRO,2020-2021,34,19.8,9.1,5.4,0.7,0.8,0.4,1.4
4369,Yovel Zoosman,Maccabi T-A,198,Forward,POL,2020-2021,11,13.5,2.6,1.5,0.6,0.0,0.5,0.5
4370,Andrey Zubkov,Zenit,206,Forward,RUS,2020-2021,39,16.0,5.1,2.1,1.1,0.0,0.3,0.9


In [25]:
df.dtypes

name          object
team          object
height         int32
position      object
country       object
Season        object
Games          int64
minutes      float64
points       float64
rebounds     float64
assists      float64
blocks       float64
steals       float64
turnovers    float64
dtype: object

In [26]:
df.isnull().sum()

name         0
team         0
height       0
position     0
country      0
Season       0
Games        0
minutes      0
points       0
rebounds     0
assists      0
blocks       0
steals       0
turnovers    0
dtype: int64

In [27]:
df

,name,team,height,position,country,Season,Games,minutes,points,rebounds,assists,blocks,steals,turnovers
0,Amadou Zaki Aboubakar,Nancy,214,Center,CMR,2008-2009,10,8.4,1.3,2.0,0.2,0.5,0.1,0.0
2,Cenk Akyol,Anadolu Efes,198,Guard,TUR,2008-2009,6,8.2,3.3,0.5,0.3,0.0,0.5,0.0
3,Sheila Alana,Avenida,191,Center,ESP,2008-2009,17,1.1,0.5,0.0,0.0,0.1,0.0,0.0
4,Tracy Albicy,Villeneuve,157,Guard,FRA,2008-2009,8,0.4,0.0,0.0,0.0,0.0,0.0,0.0
8,Audrey Alves,Villeneuve,190,Forward,FRA,2008-2009,9,0.4,0.0,0.1,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4366,Denis Zakharov,Zenit,192,Guard,RUS,2020-2021,17,5.0,0.9,0.4,0.9,0.1,0.3,0.6
4367,Paul Zipser,Bayern,203,Forward,GER,2020-2021,39,21.9,9.1,3.0,1.1,0.3,0.7,0.7
4368,Ante Zizic,Maccabi T-A,210,Center,CRO,2020-2021,34,19.8,9.1,5.4,0.7,0.8,0.4,1.4
4369,Yovel Zoosman,Maccabi T-A,198,Forward,POL,2020-2021,11,13.5,2.6,1.5,0.6,0.0,0.5,0.5


In [28]:
df.to_csv('el_players_clean.csv', index=False)